# Datamine ATTCHK Process Example

This notebook demonstrates how to configure and run the **`attchk`** process wrapper in `dmstudio`.

## Process Description

## ATTCHK

This process is used to check the legend definition file prior to running [ATTSET](<attset.md>).

**ATTCHK** copies values from previous records into blank fields, and reports any inconsistencies in the file.

A hierarchy is verified in each legend specification found in the file :-

1. Each **LEGEND** can contain one or more **DATFLD** entries

2. Each **DATFLD** can contain one or more ranges and/or patterns

3. Each range/pattern can have one or more **ATTRIB** entries.

The values of fields at each key change are propagated down until the next non-blank value in a column. One of the combinations below is expected for each **DATFIELD** /**ATTRIB** entry :-

* **DATMIN** (discrete value)

* **DATMIN** , **DATMAX** (data range)

* **DATEXP** (data pattern)

The validation of the file reports :-

1. Missing values on a record

2. Overlapping, missing or over-specified ranges

Alphanumeric range fields for which a blank is intentionally to be specified should have blanks included within single or double quotes within the legend file. A blank field will be output.

### Input Files:

* **in** (Input legend file. The following standard field names are recognized but not all need be present in the file :- LEGEND A8 Legend key. DATFIELD A8 Data field in the input file. DATMIN A12 Minimum value. DATMAX A12 Maximum value. DATEXP A40 Match (regular) expression. ATTFIELD A8 Attribute field. ATTVALUE A12 Attribute field value. Alternate field names can be supplied to the process by specification through the symbolic field names eg DATMIN(MIN).):
  Overwritten
  Required=No

### Output Files:

* **out** (Undefined):
  Output legend file. Can be the same as the input file.
  Required=No

* **error** (Undefined):
  Optional output error file for invalid records.
  Required=No

### Fields:

### Parameters:

In [ ]:
# Step 1: Connect to Datamine and Initialize Sandbox
import os
import shutil
import glob

import pandas as pd

from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()
agent.initialize_sandbox(notebook_folder)

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('attchk')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine database locally to this sandbox folder using a `t_` prefix.

In [ ]:
# Step 3: Copy VBOP datasets dynamically from Database to test_sandbox
files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

agent.copy_database_files(files_to_copy)

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute attchk
print("Running attchk...")
dm_cmd.attchk(
    # in_i='t_assays',  # optional
    # out_o='t_attchk_out',  # optional
    # error_o='t_attchk_errors',  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("attchk execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = 't_attchk_out.dmx'
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob("t_*.*")
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = '__pycache__'
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")